# Stage 2.2 - Fold-specific frozen-encoder bridge training

This notebook loads the matching fold's audited P2 encoder, freezes it, trains only the shared bi-planar fusion/projection/lift bridge through disposable multiscale 1x1x1 heads, then exports a frozen head-free front end.

Only certified training and validation samples are opened. Training uses deterministic photometric augmentation; validation and feature hashing use clean DRRs. The final checkpoint contains no supervision-head parameters.

## Scientific rationale

The comparison must isolate decoder design. The P2 encoder therefore remains byte-identical and in evaluation mode. Four scale-local heads provide direct occupancy supervision to the bridge without introducing a trainable 256-cubed neutral decoder. This follows the companion-head principle of deeply supervised networks while keeping the downstream U-style and V-style decoder comparison uncontaminated.

Local convolutional fusion is retained at 64x64 and 32x32; cross-attention is retained at 16x16 and 8x8. AP and LAT features are lifted into the validated LPS-aligned 3D grid, and targets remain ordered as `[femur, tibia, patella, fibula]`.

## Success criteria

1. The P2 encoder state hash is identical before and after bridge training, every encoder parameter has `requires_grad=False`, and the encoder remains in evaluation mode.
2. Only fusion, 2D projection, 3D fusion/lift, and disposable 1x1x1 heads receive gradients.
3. `shared_frontend.pth` contains no supervision-head parameters and records encoder, bridge, manifest, configuration, and feature-shape hashes.
4. Train, validation, and test subjects are disjoint and no test DRR or target is opened.
5. A batch-size-one forward/loss/backward step is finite and, on the HPC L4, peak allocated GPU memory remains below 90% of reported device memory.
6. Full decoder cross-validation remains blocked unless the independent Stage 2 QA status is `PASS`; audit warnings remain visible in provenance.

## Academic basis

- [Lee et al., *Deeply-Supervised Nets* (AISTATS 2015)](https://proceedings.mlr.press/v38/lee15a.html): companion objectives can directly supervise hidden representations.
- [Hatamizadeh et al., *UNETR* (WACV 2022)](https://openaccess.thecvf.com/content/WACV2022/html/Hatamizadeh_UNETR_Transformers_for_3D_Medical_Image_Segmentation_WACV_2022_paper.html): its controlled decoder-choice ablation demonstrates comparison under a common encoder.


In [ ]:
import contextlib
import hashlib
import json
import math
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")  # Must precede the torch import.
import platform
try:
    import resource
except ImportError:
    resource = None
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, get_worker_info
import timm

PROTOCOL_VERSION = "baseline_protocol_v1"
STAGE2_SCHEMA = "foundation_stage2_v1"
SEED = 42
FOLD = 0
RUN_REAL_DATA = True
RUN_FRONTEND_TRAINING = True
RESUME_FROM = None

PRETRAIN_MODEL = "convnextv2_tiny.fcmae"
IMAGE_SIZE = 256
TARGET_SIZE = 256
BONES = ["femur", "tibia", "patella", "fibula"]
OUTPUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attention", "attention"]

EPOCHS = 40
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
NUM_WORKERS = 4 if os.name != "nt" else 0
USE_AMP = True

AUGMENTATION = {
    "kind": "online_photometric_only",
    "gamma": [0.90, 1.10],
    "brightness": [-0.05, 0.05],
    "gaussian_noise_sigma": [0.0, 0.02],
    "clamp": [0.0, 1.0],
    "independent_ap_lat": True,
    "forbidden": ["crop", "rotation", "translation", "flip", "elastic", "cutout", "random_erasing"],
}

assert 0 <= FOLD < 5
assert TARGET_SIZE == 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"fold": FOLD, "device": str(DEVICE), "run_real_data": RUN_REAL_DATA})

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "configs" / "baseline_protocol_v1.json").exists() and (candidate / "reports" / "manifests").exists():
            return candidate
    raise FileNotFoundError("project root not found")


ROOT = find_project_root(Path.cwd())
MANIFEST_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.csv"
MANIFEST_META_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.metadata.json"
BASELINE_CONFIG_PATH = ROOT / "configs" / "baseline_protocol_v1.json"
DATA_CONFIG_PATH = ROOT / "configs" / "data_contract_v1.json"
FOLD_ROOT = ROOT / "models" / STAGE2_SCHEMA / f"fold_{FOLD}"
P2_ENCODER_PATH = FOLD_ROOT / "fcmae_p2_encoder.pth"
P1_CONFIG_PATH = FOLD_ROOT / "fcmae_p1_config.json"
P2_FEATURE_AUDIT_PATH = FOLD_ROOT / "feature_audit" / "p2" / "feature_audit_summary.json"


def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def canonical_sha256(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def tensor_sha256(tensor):
    value = tensor.detach().cpu().contiguous()
    h = hashlib.sha256()
    h.update(str(value.dtype).encode())
    h.update(np.asarray(value.shape, dtype=np.int64).tobytes())
    h.update(value.numpy().tobytes())
    return h.hexdigest()


def state_sha256(state):
    h = hashlib.sha256()
    for key in sorted(state):
        h.update(key.encode()); h.update(tensor_sha256(state[key]).encode())
    return h.hexdigest()


def stable_seed(*parts):
    token = "|".join(str(p) for p in parts).encode()
    return int.from_bytes(hashlib.sha256(token).digest()[:8], "little") % (2**32)


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def augment_drr(array, sample_id, view, epoch, worker_id):
    rng = np.random.default_rng(stable_seed(SEED, FOLD, epoch, worker_id, sample_id, view))
    gamma = rng.uniform(*AUGMENTATION["gamma"])
    brightness = rng.uniform(*AUGMENTATION["brightness"])
    sigma = rng.uniform(*AUGMENTATION["gaussian_noise_sigma"])
    output = np.power(np.clip(array, 0, 1), gamma, dtype=np.float32) + np.float32(brightness)
    if sigma > 0: output += rng.normal(0, sigma, output.shape).astype(np.float32)
    return np.clip(output, 0, 1).astype(np.float32)


def load_certified_folds(fold):
    meta = json.loads(MANIFEST_META_PATH.read_text(encoding="utf-8"))
    baseline = json.loads(BASELINE_CONFIG_PATH.read_text(encoding="utf-8"))
    data_contract = json.loads(DATA_CONFIG_PATH.read_text(encoding="utf-8"))
    if baseline["protocol_version"] != PROTOCOL_VERSION: raise RuntimeError("protocol mismatch")
    if not meta.get("certification_approved", False): raise RuntimeError("Stage 1 is not certified")
    if int(meta.get("ready_rows", -1)) != 71 or int(meta.get("pending_recertification_rows", -1)) != 0: raise RuntimeError("Stage 1 must have 71 ready and zero pending rows")
    leakage = meta.get("leakage", {})
    if any(int(leakage.get(field, -1)) != 0 for field in ("subject_multiple_test_folds", ("fold" + str(5) + "_rows"), "augmentation_parent_mismatch")): raise RuntimeError(f"certified leakage report is not zero: {leakage}")
    if sha256_file(MANIFEST_PATH) != meta.get("sha256"): raise RuntimeError("manifest hash mismatch")
    rows = pd.read_csv(MANIFEST_PATH, dtype={"test_fold": "Int64"})
    rows = rows[rows.status.eq("ready")].copy()
    if len(rows) != 71 or rows.groupby("dataset").size().to_dict() != {"Ruikar": 13, "VSD": 58}: raise RuntimeError("certified cohort mismatch")
    rows["test_fold"] = rows.test_fold.astype(int)
    if set(rows.test_fold) != set(range(5)) or rows.groupby("subject_id").test_fold.nunique().max() != 1: raise RuntimeError("fold isolation failure")
    parent_rows = rows.set_index("sample_id")[["subject_id", "test_fold"]]
    for row in rows.itertuples(index=False):
        if row.augmentation_parent not in parent_rows.index: raise RuntimeError(f"missing augmentation parent: {row.sample_id}")
        parent = parent_rows.loc[row.augmentation_parent]
        if parent["subject_id"] != row.subject_id or int(parent["test_fold"]) != int(row.test_fold): raise RuntimeError(f"augmentation leakage: {row.sample_id}")
    if rows.target_version.ne(data_contract["target_version"]).any() or rows.drr_version.ne(data_contract["drr_version"]).any(): raise RuntimeError("data version mismatch")
    rows["split"] = "train"
    rows.loc[rows.test_fold.eq(fold), "split"] = "test"
    rows.loc[rows.test_fold.eq((fold + 1) % 5), "split"] = "validation"
    by_split = {name: set(group.subject_id) for name, group in rows.groupby("split")}
    if by_split["train"] & by_split["validation"] or by_split["train"] & by_split["test"] or by_split["validation"] & by_split["test"]: raise RuntimeError("subject overlap between roles")
    for split in ("train", "validation"):
        for row in rows[rows.split.eq(split)].itertuples(index=False):
            for field in ("ap_drr_path", "lat_drr_path"):
                if not (ROOT / getattr(row, field)).is_file(): raise FileNotFoundError(f"missing certified {split} DRR: {getattr(row, field)}")
            target_dir = ROOT / row.target_path
            for bone in BONES:
                if not (target_dir / f"{row.sample_id}_{bone}.nii.gz").is_file(): raise FileNotFoundError(f"missing certified {split} target: {row.sample_id}/{bone}")
    return rows[rows.split.eq("train")].copy(), rows[rows.split.eq("validation")].copy(), rows[rows.split.eq("test")].copy(), meta


def read_drr(path):
    array = np.load(path).astype(np.float32)
    if array.shape != (256, 256) or not np.isfinite(array).all() or array.min() < -1e-6 or array.max() > 1 + 1e-6: raise ValueError(f"invalid DRR: {path}")
    return np.clip(array, 0, 1)


def load_target(row):
    target_dir = ROOT / row.target_path
    arrays, geometry = [], None
    for bone in BONES:
        image = nib.load(str(target_dir / f"{row.sample_id}_{bone}.nii.gz"))
        if image.shape != (256, 256, 256): raise ValueError(f"target shape mismatch: {row.sample_id}/{bone}")
        if tuple(nib.aff2axcodes(image.affine)) != ("L", "P", "S"): raise ValueError(f"target orientation mismatch: {row.sample_id}/{bone}")
        current_geometry = (tuple(np.round(image.affine.ravel(), 7)), tuple(np.round(image.header.get_zooms()[:3], 7)))
        if geometry is None: geometry = current_geometry
        if current_geometry != geometry: raise ValueError(f"per-bone geometry mismatch: {row.sample_id}")
        array = np.asarray(image.dataobj, dtype=np.float32)
        unique = np.unique(array)
        if not set(unique.tolist()).issubset({0.0, 1.0}) or array.sum() == 0: raise ValueError(f"non-binary or empty target: {row.sample_id}/{bone}")
        arrays.append(array)
    return torch.from_numpy(np.stack(arrays, axis=0).astype(np.float32))


class FrontEndDataset(Dataset):
    def __init__(self, rows, training):
        self.rows = rows.reset_index(drop=True); self.training = training; self.epoch = 0
    def set_epoch(self, epoch): self.epoch = int(epoch)
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        worker = get_worker_info(); worker_id = 0 if worker is None else worker.id
        ap = read_drr(ROOT / row.ap_drr_path); lat = read_drr(ROOT / row.lat_drr_path)
        if self.training:
            ap = augment_drr(ap, row.sample_id, "ap", self.epoch, worker_id)
            lat = augment_drr(lat, row.sample_id, "lat", self.epoch, worker_id)
        return {"ap": torch.from_numpy(ap).unsqueeze(0), "lat": torch.from_numpy(lat).unsqueeze(0), "target": load_target(row), "sample_id": row.sample_id, "subject_id": row.subject_id}


seed_everything()
print("project root:", ROOT)

In [ ]:
class CrossAttention(nn.Module):
    """Use low-resolution tokens to exchange global AP/LAT context at tractable memory cost."""
    def __init__(self, dim):
        super().__init__(); self.query = nn.Linear(dim, dim); self.key = nn.Linear(dim, dim); self.value = nn.Linear(dim, dim); self.scale = dim ** -0.5
    def forward(self, query_map, context_map):
        batch, channels, height, width = query_map.shape
        query = query_map.flatten(2).transpose(1, 2); context = context_map.flatten(2).transpose(1, 2)
        attention = torch.softmax(self.query(query) @ self.key(context).transpose(-2, -1) * self.scale, dim=-1)
        return (attention @ self.value(context) + query).transpose(1, 2).reshape(batch, channels, height, width)


class LocalFusion(nn.Module):
    """Preserve high-resolution neighbourhood detail with a small residual convolutional mixer."""
    def __init__(self, dim):
        super().__init__(); self.mix = nn.Conv2d(2 * dim, dim, 3, padding=1)
    def forward(self, query_map, context_map): return self.mix(torch.cat([query_map, context_map], dim=1)) + query_map


class BiPlanarFrontEnd(nn.Module):
    """Shared encoder plus scale-dependent bidirectional fusion and geometry-locked 3D lifting."""
    def __init__(self, encoder_state, pretrained_configuration):
        super().__init__()
        self.encoder = timm.create_model(PRETRAIN_MODEL, pretrained=False, features_only=True)
        incompatible = self.encoder.load_state_dict(encoder_state, strict=True)
        if incompatible.missing_keys or incompatible.unexpected_keys: raise RuntimeError(f"P2 encoder strict-load failure: {incompatible}")
        self.pretrained_configuration = pretrained_configuration
        feature_channels = self.encoder.feature_info.channels()
        self.fusion = nn.ModuleList([CrossAttention(c) if kind == "attention" else LocalFusion(c) for c, kind in zip(feature_channels, FUSION_TYPES)])
        self.project_2d = nn.ModuleList([nn.Conv2d(source, target, 1) for source, target in zip(feature_channels, OUTPUT_CHANNELS)])
        self.fuse_3d = nn.ModuleList([nn.Conv3d(2 * channels, channels, 3, padding=1) for channels in OUTPUT_CHANNELS])

    @staticmethod
    def _orthogonal_lift(ap_feature, lat_feature, projection, fusion3d):
        """Expand each projection along its missing ray axis, then learn only their local 3D combination."""
        if ap_feature.shape != lat_feature.shape or ap_feature.ndim != 4 or ap_feature.shape[-2] != ap_feature.shape[-1]:
            raise RuntimeError(f"AP/LAT lift geometry mismatch: {tuple(ap_feature.shape)} vs {tuple(lat_feature.shape)}")
        ap = projection(ap_feature); lat = projection(lat_feature).flip(3)
        batch, channels, size, _ = ap.shape
        # AP contributes the coronal pattern through depth; LAT contributes the sagittal pattern through width.
        # The lateral flip is fixed by the validated LPS convention, not learned from supervision.
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(batch, channels, size, size, size)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(batch, channels, size, size, size)
        return fusion3d(torch.cat([ap_cube, lat_cube], dim=1))

    def normalize(self, raw):
        image = raw.repeat(1, 3, 1, 1)
        mean = torch.as_tensor(self.pretrained_configuration["mean"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1)
        std = torch.as_tensor(self.pretrained_configuration["std"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1)
        return (image - mean) / std

    def forward(self, ap_raw, lat_raw):
        if ap_raw.shape != lat_raw.shape or tuple(ap_raw.shape[-2:]) != (TARGET_SIZE, TARGET_SIZE):
            raise RuntimeError(f"AP/LAT input geometry mismatch: {tuple(ap_raw.shape)} vs {tuple(lat_raw.shape)}")
        ap_levels = self.encoder(self.normalize(ap_raw)); lat_levels = self.encoder(self.normalize(lat_raw)); lifted = []
        for level, (ap, lat, fusion, project, fuse3d) in enumerate(zip(ap_levels, lat_levels, self.fusion, self.project_2d, self.fuse_3d)):
            expected_size = TARGET_SIZE // (4 * (2 ** level))
            if ap.shape != lat.shape or tuple(ap.shape[-2:]) != (expected_size, expected_size):
                raise RuntimeError(f"AP/LAT feature geometry mismatch at level {level}: {tuple(ap.shape)} vs {tuple(lat.shape)}")
            ap_refined = fusion(ap, lat); lat_refined = fusion(lat, ap)
            lifted.append(self._orthogonal_lift(ap_refined, lat_refined, project, fuse3d))
        return lifted


def dice_bce_loss(logits, target):
    """Balance voxel calibration (BCE) with overlap quality (soft Dice) across four bones."""
    logits = logits.float(); target = target.float(); bce = F.binary_cross_entropy_with_logits(logits, target)
    probability = torch.sigmoid(logits).flatten(2); flattened_target = target.flatten(2); intersection = (probability * flattened_target).sum(-1)
    dice = (2 * intersection + 1.0) / (probability.sum(-1) + flattened_target.sum(-1) + 1.0)
    return 0.5 * bce + 0.5 * (1.0 - dice.mean())


@torch.no_grad()
def hard_per_bone_dice(logits, target):
    prediction = (torch.sigmoid(logits.float()) > 0.5).float().flatten(2); target = (target > 0.5).float().flatten(2); intersection = (prediction * target).sum(-1)
    return (2 * intersection + 1e-6) / (prediction.sum(-1) + target.sum(-1) + 1e-6)

In [ ]:
def amp_context():
    if not (USE_AMP and DEVICE.type == "cuda"): return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)


def make_scaler(): return torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported())


def capture_rng_state(): return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng_state(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"]); torch.set_rng_state(state["torch"])
    if state.get("cuda") is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(state["cuda"])


def require_p2_feature_audit():
    """Require trustworthy audit execution while allowing scientific quality warnings to propagate."""
    if not P2_FEATURE_AUDIT_PATH.is_file():
        raise FileNotFoundError(f"completed P2 feature audit is required: {P2_FEATURE_AUDIT_PATH}")
    summary = json.loads(P2_FEATURE_AUDIT_PATH.read_text(encoding="utf-8"))
    if summary.get("stage") != "P2" or summary.get("fold") != FOLD:
        raise RuntimeError("P2 feature-audit stage/fold mismatch")
    if summary.get("structural_status") != "PASS" or not summary.get("encoder_unchanged"):
        raise RuntimeError("P2 feature-audit structural gate failed")
    if summary.get("test_paths_opened") is not False:
        raise RuntimeError("P2 feature audit does not prove test paths remained unopened")
    return summary

def build_front_end():
    require_p2_feature_audit()
    if not P2_ENCODER_PATH.is_file() or not P1_CONFIG_PATH.is_file(): raise FileNotFoundError("matching fold P2 encoder/provenance is required")
    export = torch.load(P2_ENCODER_PATH, map_location="cpu", weights_only=False)
    if export.get("schema_version") != STAGE2_SCHEMA or export.get("fold") != FOLD or export.get("stage") != "cross_view_P2": raise RuntimeError("P2 export schema/fold/stage mismatch")
    p1_config = json.loads(P1_CONFIG_PATH.read_text(encoding="utf-8")); pretrained_configuration = p1_config.get("pretrained_configuration")
    if not pretrained_configuration: raise RuntimeError("P1 config does not record pretrained_configuration")
    return BiPlanarFrontEnd(export["encoder_state"], pretrained_configuration), export


def resource_usage(started):
    peak_host = None
    if resource is not None:
        maximum_rss = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        peak_host = maximum_rss if platform.system() == "Darwin" else maximum_rss * 1024
    peak_gpu = int(torch.cuda.max_memory_allocated()) if DEVICE.type == "cuda" else None
    total_gpu = int(torch.cuda.get_device_properties(DEVICE).total_memory) if DEVICE.type == "cuda" else None
    return {"wall_seconds": round(time.time() - started, 1), "peak_gpu_bytes": peak_gpu, "total_gpu_bytes": total_gpu, "peak_host_bytes": peak_host}


def save_frontend_curve(history, path):
    frame = pd.DataFrame(history); figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(frame.epoch, frame.train_loss, label="train"); axes[0].plot(frame.epoch, frame.validation_loss, label="validation"); axes[0].set(title=f"Fold {FOLD} loss", xlabel="epoch"); axes[0].legend(); axes[0].grid(alpha=0.25)
    for bone in BONES: axes[1].plot(frame.epoch, frame[f"validation_dice_{bone}"], label=bone)
    axes[1].set(title="Validation hard Dice", xlabel="epoch", ylim=(0, 1)); axes[1].legend(); axes[1].grid(alpha=0.25)
    figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def frontend_config(manifest_meta, train_rows, validation_rows, test_rows, p2_sha):
    return {"schema_version": STAGE2_SCHEMA, "protocol_version": PROTOCOL_VERSION, "stage": "frozen_encoder_shared_bridge", "fold": FOLD, "seed": SEED, "manifest_sha256": manifest_meta["sha256"], "p2_encoder_sha256": p2_sha, "train_sample_ids": sorted(train_rows.sample_id.tolist()), "validation_sample_ids": sorted(validation_rows.sample_id.tolist()), "test_sample_ids_not_opened": sorted(test_rows.sample_id.tolist()), "augmentation": AUGMENTATION, "target": "four_channel_binary_occupancy", "bones": BONES, "architecture": {"fusion_types": FUSION_TYPES, "lift": "bidirectional_hybrid_orthogonal_lps", "feature_channels": OUTPUT_CHANNELS, "neutral_block": "single_conv_groupnorm8_relu", "neutral_upsampling": "trilinear_align_corners_false", "neutral_checkpoint_scope": "upsample_and_refine", "neutral_head_discarded": True}, "hyperparameters": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "optimizer": "Adam", "learning_rate": LEARNING_RATE, "loss": "0.5_bce+0.5_soft_dice", "amp": USE_AMP, "activation_checkpointing": False}, "software": {"python": platform.python_version(), "torch": torch.__version__, "timm": timm.__version__, "nibabel": nib.__version__}, "hardware": {"device": str(DEVICE), "cuda_device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None, "cuda_allocator": os.environ.get("PYTORCH_ALLOC_CONF")}, "output_paths": {"artifact_root": str(FOLD_ROOT), "front_end_checkpoint": str(FOLD_ROOT / "shared_frontend.pth")}}


def save_trainstate(path, epoch, global_step, model, optimizer, scheduler, scaler, best_dice, best_loss, config_sha):
    torch.save({"schema_version": STAGE2_SCHEMA, "stage": "frozen_encoder_shared_bridge", "fold": FOLD, "epoch": epoch, "global_step": global_step, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "rng_state": capture_rng_state(), "best_macro_dice": best_dice, "best_validation_loss": best_loss, "config_sha256": config_sha}, path)


def strict_load(module, state, label):
    incompatible = module.load_state_dict(state, strict=True)
    if incompatible.missing_keys or incompatible.unexpected_keys: raise RuntimeError(f"{label} strict-load failed: {incompatible}")


In [ ]:
# Controlled bridge-training override: freezes P2 and replaces the 256^3 neutral decoder.
BRIDGE_PATIENCE = 8


class BridgeSupervisionHeads(nn.Module):
    """Disposable 1x1x1 heads; they never enter the exported front end."""
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([nn.Conv3d(channels, len(BONES), 1) for channels in OUTPUT_CHANNELS])

    def forward(self, features):
        if len(features) != len(self.heads):
            raise RuntimeError("bridge feature/head count mismatch")
        return [head(feature) for head, feature in zip(self.heads, features)]


class ControlledBridgeTrainingModel(nn.Module):
    """Train bridge plus disposable heads while P2 stays frozen and in eval mode."""
    def __init__(self, front_end):
        super().__init__()
        self.front_end = front_end
        self.supervision_heads = BridgeSupervisionHeads()
        for parameter in self.front_end.encoder.parameters():
            parameter.requires_grad = False
        self.front_end.encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        self.front_end.encoder.eval()
        return self

    def forward(self, ap, lat):
        return self.supervision_heads(self.front_end(ap, lat))


def pooled_target(target, logits):
    """Max pooling retains small/thin bone occupancy at every supervised scale."""
    return F.adaptive_max_pool3d(target.float(), output_size=logits.shape[-3:])


def multiscale_bridge_loss(logits_by_scale, target):
    losses = [dice_bce_loss(logits, pooled_target(target, logits)) for logits in logits_by_scale]
    return torch.stack(losses).mean()


@torch.no_grad()
def multiscale_hard_dice(logits_by_scale, target):
    values = [hard_per_bone_dice(logits, pooled_target(target, logits)) for logits in logits_by_scale]
    return torch.stack(values, dim=0).mean(dim=0)


def run_bridge_epoch(model, loader, optimizer, scaler, training):
    model.train(training)
    total_loss, count, dice_rows = 0.0, 0, []
    for batch_index, batch in enumerate(loader):
        if training:
            optimizer.zero_grad(set_to_none=True)
        ap = batch["ap"].to(DEVICE, non_blocking=True)
        lat = batch["lat"].to(DEVICE, non_blocking=True)
        target = batch["target"].to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training):
            with amp_context():
                logits_by_scale = model(ap, lat)
                loss = multiscale_bridge_loss(logits_by_scale, target)
        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite frozen-encoder bridge loss")
        if training:
            scaler.scale(loss).backward()
            if batch_index == 0:
                if any(parameter.grad is not None for parameter in model.front_end.encoder.parameters()):
                    raise RuntimeError("frozen P2 encoder received gradients")
                trainable_bridge = list(model.front_end.fusion.parameters()) + list(model.front_end.project_2d.parameters()) + list(model.front_end.fuse_3d.parameters())
                if not any(parameter.grad is not None for parameter in trainable_bridge):
                    raise RuntimeError("bridge did not receive gradients")
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * ap.shape[0]
        count += ap.shape[0]
        dice_rows.append(multiscale_hard_dice(logits_by_scale, target).cpu())
    return total_loss / count, torch.cat(dice_rows, dim=0).mean(dim=0).numpy()


@torch.no_grad()
def verify_frozen_frontend(front_end, rows):
    """Use one clean validation case to prove deterministic feature shapes and hashes."""
    item = FrontEndDataset(rows.iloc[:1], training=False)[0]
    ap = item["ap"].unsqueeze(0).to(DEVICE)
    lat = item["lat"].unsqueeze(0).to(DEVICE)
    first = [value.float().cpu().contiguous() for value in front_end(ap, lat)]
    second = [value.float().cpu().contiguous() for value in front_end(ap, lat)]
    hashes = [tensor_sha256(value) for value in first]
    if hashes != [tensor_sha256(value) for value in second]:
        raise RuntimeError("frozen front end is not deterministic")
    required = [[1, 64, 64, 64, 64], [1, 128, 32, 32, 32], [1, 256, 16, 16, 16], [1, 512, 8, 8, 8]]
    shapes = [list(value.shape) for value in first]
    if shapes != required:
        raise RuntimeError(f"frozen feature shape mismatch: {shapes}")
    return {"sample_id": item["sample_id"], "feature_shapes": shapes, "feature_sha256": hashes}


def train_frontend(train_rows, validation_rows, test_rows, manifest_meta):
    """Train only the fold-specific bridge; P2 and test-fold data remain untouched."""
    FOLD_ROOT.mkdir(parents=True, exist_ok=True)
    started = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    audit_summary = require_p2_feature_audit()
    front_end, _ = build_front_end()
    p2_file_sha = sha256_file(P2_ENCODER_PATH)
    p2_state_sha_before = state_sha256(front_end.encoder.state_dict())
    model = ControlledBridgeTrainingModel(front_end).to(DEVICE)
    if any(parameter.requires_grad for parameter in model.front_end.encoder.parameters()):
        raise RuntimeError("P2 encoder was not frozen")
    config = frontend_config(manifest_meta, train_rows, validation_rows, test_rows, p2_file_sha)
    config["stage"] = "frozen_encoder_shared_bridge"
    config["architecture"] = {
        "fusion_types": FUSION_TYPES,
        "lift": "bidirectional_hybrid_orthogonal_lps",
        "feature_channels": OUTPUT_CHANNELS,
        "p2_encoder": "frozen_eval",
        "supervision": "four_disposable_1x1x1_multiscale_heads",
        "target_downsampling": "adaptive_max_pool3d",
        "supervision_heads_discarded": True,
    }
    config["hyperparameters"].update({"patience": BRIDGE_PATIENCE, "selection": "validation_multiscale_macro_dice", "activation_checkpointing": False})
    config["p2_feature_audit"] = {
        "sha256": sha256_file(P2_FEATURE_AUDIT_PATH),
        "structural_status": audit_summary.get("structural_status"),
        "quality_status": audit_summary.get("quality_status"),
        "warnings": audit_summary.get("warnings", []),
        "full_cv_blocked": audit_summary.get("quality_status") != "PASS",
    }
    config_sha = canonical_sha256(config)
    (FOLD_ROOT / "shared_frontend_config.json").write_text(json.dumps(config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = make_scaler()
    train_set = FrontEndDataset(train_rows, training=True)
    validation_set = FrontEndDataset(validation_rows, training=False)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    start_epoch, global_step, best_dice, best_loss, stale, history = 0, 0, -1.0, float("inf"), 0, []
    if RESUME_FROM:
        saved = torch.load(RESUME_FROM, map_location=DEVICE, weights_only=False)
        if saved["fold"] != FOLD or saved["config_sha256"] != config_sha:
            raise RuntimeError("bridge resume fold/config mismatch")
        strict_load(model, saved["model"], "bridge training model")
        optimizer.load_state_dict(saved["optimizer"]); scheduler.load_state_dict(saved["scheduler"]); scaler.load_state_dict(saved["scaler"]); restore_rng_state(saved["rng_state"])
        start_epoch, global_step, best_dice, best_loss = saved["epoch"] + 1, saved["global_step"], saved["best_macro_dice"], saved["best_validation_loss"]
    for epoch in range(start_epoch, EPOCHS):
        train_set.set_epoch(epoch)
        train_loss, train_dice = run_bridge_epoch(model, train_loader, optimizer, scaler, True)
        global_step += len(train_loader)
        validation_loss, validation_dice = run_bridge_epoch(model, validation_loader, optimizer, scaler, False)
        scheduler.step()
        macro = float(validation_dice.mean())
        record = {"epoch": epoch, "train_loss": train_loss, "validation_loss": validation_loss, "train_macro_dice": float(train_dice.mean()), "validation_macro_dice": macro, **{f"validation_dice_{bone}": float(validation_dice[i]) for i, bone in enumerate(BONES)}, "lr": optimizer.param_groups[0]["lr"], "elapsed_seconds": round(time.time() - started, 1)}
        history.append(record)
        pd.DataFrame(history).to_csv(FOLD_ROOT / "shared_frontend_history.csv", index=False)
        improved = macro > best_dice or (math.isclose(macro, best_dice) and validation_loss < best_loss)
        if improved:
            best_dice, best_loss, stale = macro, validation_loss, 0
            save_trainstate(FOLD_ROOT / "shared_frontend_best_trainstate.pth", epoch, global_step, model, optimizer, scheduler, scaler, best_dice, best_loss, config_sha)
        else:
            stale += 1
        save_trainstate(FOLD_ROOT / "shared_frontend_last_trainstate.pth", epoch, global_step, model, optimizer, scheduler, scaler, best_dice, best_loss, config_sha)
        print(record)
        if stale >= BRIDGE_PATIENCE:
            print(f"early stopping after {BRIDGE_PATIENCE} non-improving epochs")
            break
    save_frontend_curve(history, FOLD_ROOT / "shared_frontend_training_curve.png")
    best = torch.load(FOLD_ROOT / "shared_frontend_best_trainstate.pth", map_location="cpu", weights_only=False)
    strict_load(model.cpu(), best["model"], "best frozen-encoder bridge model")
    p2_state_sha_after = state_sha256(model.front_end.encoder.state_dict())
    if p2_state_sha_before != p2_state_sha_after:
        raise RuntimeError("P2 encoder changed during bridge training")
    front_end_state = {key: value.cpu() for key, value in model.front_end.state_dict().items()}
    if any("supervision_heads" in key for key in front_end_state):
        raise RuntimeError("disposable supervision head leaked into export")
    export = {"schema_version": STAGE2_SCHEMA, "stage": "shared_frontend", "fold": FOLD, "front_end_state": front_end_state, "config_sha256": config_sha, "manifest_sha256": manifest_meta["sha256"], "p2_encoder_sha256": p2_file_sha, "p2_encoder_state_sha256": p2_state_sha_after}
    export_path = FOLD_ROOT / "shared_frontend.pth"
    torch.save(export, export_path)
    reloaded, _ = build_front_end()
    strict_load(reloaded, export["front_end_state"], "head-free shared front end")
    for parameter in reloaded.parameters():
        parameter.requires_grad = False
    reloaded = reloaded.to(DEVICE).eval()
    feature_evidence = verify_frozen_frontend(reloaded, validation_rows)
    usage = resource_usage(started)
    if usage["peak_gpu_bytes"] is not None and usage["peak_gpu_bytes"] >= 0.90 * usage["total_gpu_bytes"]:
        raise RuntimeError("bridge training used at least 90% of GPU memory")
    provenance = {**config, "config_sha256": config_sha, "best_validation_macro_dice": best_dice, "best_validation_loss": best_loss, "p2_encoder_state_sha256_before": p2_state_sha_before, "p2_encoder_state_sha256_after": p2_state_sha_after, "front_end_state_sha256": state_sha256(front_end_state), "checkpoint_sha256": sha256_file(export_path), "feature_evidence": feature_evidence, "resource_usage": usage, "success": p2_state_sha_before == p2_state_sha_after and all(not p.requires_grad for p in reloaded.parameters()), "qa_figures": [str(FOLD_ROOT / "shared_frontend_training_curve.png")]}
    (FOLD_ROOT / "shared_frontend_provenance.json").write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return provenance


In [ ]:
if RUN_REAL_DATA:
    train_rows, validation_rows, test_rows, manifest_meta = load_certified_folds(FOLD)
    print({"train": len(train_rows), "validation": len(validation_rows), "test_not_opened": len(test_rows)})
    if RUN_FRONTEND_TRAINING: print(train_frontend(train_rows, validation_rows, test_rows, manifest_meta))
else:
    print("DATA-FREE MODE: front-end definitions loaded. Real training requires a structurally valid P2 feature audit.")